# Day 06 - Nsight Compute profiling analysis

Day 05 built five transpose variants. Day 06 asks two questions:

> Why is the naive coalesced-write mapping much faster than the
> naive coalesced-read mapping?

> Do padding and `S<5,0,5>` swizzling remove the shared-memory bank
> conflict in the partitioned Challenge B kernel?

You will collect five controlled Nsight Compute reports and connect
global sectors, shared bank conflicts, and duration to the layouts.

## 1. Begin with a falsifiable hypothesis

Challenge A predicts:

```text
naive-read:  few load sectors/request, many store sectors/request
naive-write: many load sectors/request, few store sectors/request
```

Challenge B predicts:

```text
unpadded: (32,32):(32,1)          high shared-load conflicts
padded:   (32,32):(33,1)          low shared-load conflicts
swizzled: S<5,0,5> o (32,32)      low conflicts, no padding
```

Before profiling Challenge B, predict:

| Metric | Prediction |
| --- | --- |
| Shared-load bank conflicts | Large decrease |
| Duration | Decrease |
| DRAM throughput % | Increase because shared serialization falls |
| Registers/thread | No material change |
| Occupancy | Nearly unchanged |
| Dynamic shared memory | Increase by about 124 bytes/CTA |

This table makes the experiment falsifiable. If conflicts do not
fall, either the bank model was wrong, the compiler changed the
access, or the wrong kernel was captured.

## 2. Control every variable except the tested mapping

A meaningful A/B profile requires:

- identical matrix shape and dtype;
- identical launch grid and block;
- identical warmup and correctness behavior;
- the same NCU version and metric list;
- exactly one captured target launch;
- no unrelated PyTorch kernels in the report.

`profile_target.py` selects one maintained Day 05 JIT function per
process. All variants use the same shape, dtype, and 256-thread
block. Each process compiles and launches exactly one target kernel.

In [ ]:
from pathlib import Path

day_root = Path.cwd()
if not (day_root / "fixtures").exists():
    day_root = day_root / "day06_ncu_profiling"

fixtures = {
    "naive read": day_root / "fixtures" / "naive_read.csv",
    "naive write": day_root / "fixtures" / "naive_write.csv",
    "shared unpadded": day_root / "fixtures" / "shared_unpadded.csv",
    "shared padded": day_root / "fixtures" / "shared_padded.csv",
    "shared swizzled": day_root / "fixtures" / "shared_swizzled.csv",
}
unpadded_path = fixtures["shared unpadded"]
reports_dir = day_root / "reports"
for name, path in fixtures.items():
    print(name, path)

## 3. Collect all five target kernels

Run profiling from a terminal on the H100 pod, not inside a busy
notebook kernel:

```bash
cd /tmp/gpu-programming-puzzle-cute-dsl/day06_ncu_profiling
PYTHON_BIN=../.venv/bin/python M=4096 N=4096               bash collect_profiles.sh
```

The helper creates:

```text
naive_read.csv             naive_write.csv
shared_unpadded.csv        shared_padded.csv
shared_swizzled.csv
```

The key NCU controls are:

```text
--kernel-name regex:<matching naive/shared kernel>
--launch-count 1
--metrics <small explicit list>
--force-overwrite
```

Kernel filtering excludes allocations and PyTorch setup kernels.
One launch keeps the report unambiguous. A small explicit metric
list minimizes replay passes and collection overhead.

## 4. Know what NCU replay means

Hardware counters cannot always be collected simultaneously. NCU
may save state and replay a kernel for multiple passes. This has
two consequences:

1. Profiled duration is perturbed and is not your production
   benchmark.
2. Metrics from one report still describe the same captured launch,
   provided replay is safe and deterministic.

Transpose writes its complete output and has no cross-launch
accumulation, so replay is safe. For kernels with atomics, random
state, inter-kernel dependencies, or persistent side effects,
replay requires additional care.

## 5. Export the raw page

`.ncu-rep` is the authoritative report for the GUI. The challenge
consumes a machine-readable raw page:

```bash
ncu --import reports/shared_unpadded.ncu-rep --csv --page raw               > reports/shared_unpadded.csv
```

NCU 2025 emits a **wide CSV**:

1. header row with metric names;
2. units row;
3. one row per captured kernel.

Do not hardcode column positions. NCU versions and requested metric
sets can add columns. Find values by metric name.

In [ ]:
import csv

rows = list(csv.reader(unpadded_path.read_text().splitlines()))
header = rows[0]
units = rows[1]
data = rows[2]

for metric in (
    "gpu__time_duration.sum",
    "l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum",
    "launch__shared_mem_per_block_dynamic",
):
    index = header.index(metric)
    print(metric, data[index], units[index])

## 6. Normalize units before calculating

NCU chooses human-readable units. A duration may be `ns`, `us`, or
`ms`; memory can appear as `byte`, `Kbyte`, or `Mbyte`. Launch
metrics can add suffixes such as `Kbyte/block`.

Convert to one internal system:

- duration: microseconds;
- DRAM traffic: decimal megabytes;
- shared storage: decimal kilobytes;
- ratios: percent;
- conflicts and registers: raw counts.

Decimal prefixes match NCU's displayed units. Unit conversion is
part of correctness: a 1000x mistake can produce a plausible-looking
but false conclusion.

## 7. Metrics answer different questions

| Metric family | Question |
| --- | --- |
| `gpu__time_duration` | Did the variant execute faster under the same collection? |
| global requests and sectors | How many 32-byte sectors did each load/store request touch? |
| average sectors/request | Which global direction is coalesced or scattered? |
| shared bank conflicts | Did padding affect the intended mechanism? |
| DRAM bytes | Did both reports perform comparable global work? |
| DRAM throughput % | Could the kernel issue memory work more effectively? |
| achieved occupancy | Did residency materially change? |
| registers and shared memory | What resource tradeoff did padding introduce? |

Duration is an outcome. Global sectors explain Challenge A; bank
conflicts explain Challenge B. Occupancy and resources are controls.

## 8. Challenge: implement the report analyzer

Fill seven TODOs:

1. Compute total bank conflicts.
2. Parse numeric fields including thousands separators.
3. Locate raw CSV header, units, and data rows.
4. Normalize units.
5. Compute shared-memory speedup and metric/resource deltas.
6. Compare the naive mappings through global sector ratios.
7. Print diagnoses supported by several counters.

Begin with the checked-in H100 fixtures, then point your analyzer at
reports you collect yourself.

In [ ]:
#!/usr/bin/env python3
# SPDX-License-Identifier: BSD-3-Clause
"""Day 06: fill the Nsight Compute report-analysis TODOs."""

from __future__ import annotations

import argparse
import csv
from dataclasses import dataclass
from pathlib import Path


METRICS = {
    "duration": "gpu__time_duration.sum",
    "dram_read": "dram__bytes_read.sum",
    "dram_write": "dram__bytes_write.sum",
    "dram_throughput": "dram__throughput.avg.pct_of_peak_sustained_elapsed",
    "global_load_requests": "l1tex__t_requests_pipe_lsu_mem_global_op_ld.sum",
    "global_store_requests": "l1tex__t_requests_pipe_lsu_mem_global_op_st.sum",
    "global_load_sectors": "l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum",
    "global_store_sectors": "l1tex__t_sectors_pipe_lsu_mem_global_op_st.sum",
    "load_sectors_per_request": (
        "l1tex__average_t_sectors_per_request_pipe_lsu_mem_global_op_ld.ratio"
    ),
    "store_sectors_per_request": (
        "l1tex__average_t_sectors_per_request_pipe_lsu_mem_global_op_st.ratio"
    ),
    "load_conflicts": (
        "l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum"
    ),
    "store_conflicts": (
        "l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum"
    ),
    "occupancy": "sm__warps_active.avg.pct_of_peak_sustained_active",
    "registers": "launch__registers_per_thread",
    "shared_memory": "launch__shared_mem_per_block_dynamic",
}


@dataclass(frozen=True)
class Profile:
    name: str
    kernel_name: str
    block_size: str
    grid_size: str
    duration_us: float
    dram_read_mb: float
    dram_write_mb: float
    dram_throughput_pct: float
    global_load_requests: float
    global_store_requests: float
    global_load_sectors: float
    global_store_sectors: float
    load_sectors_per_request: float
    store_sectors_per_request: float
    load_bank_conflicts: float
    store_bank_conflicts: float
    occupancy_pct: float
    registers_per_thread: float
    dynamic_shared_kb: float

    @property
    def total_bank_conflicts(self) -> float:
        # TODO(1): Return load plus store bank conflicts.
        raise NotImplementedError("Day 06 TODO(1): total bank conflicts")


def parse_number(text: str) -> float:
    # TODO(2): Strip whitespace and thousands separators, reject empty/N/A,
    # and return a float.
    raise NotImplementedError("Day 06 TODO(2): parse NCU numeric field")


def load_raw_csv(path: Path) -> tuple[dict[str, str], dict[str, str]]:
    rows = list(csv.reader(path.read_text(encoding="utf-8").splitlines()))

    # TODO(3): Find the wide raw-page header containing both "Kernel Name" and
    # METRICS["duration"]. The next row contains units and the following
    # non-empty row contains kernel values. Return:
    #
    #   dict(zip(header, data)), dict(zip(header, units))
    #
    # Pad short units/data rows with empty strings before zipping.
    raise NotImplementedError("Day 06 TODO(3): parse NCU raw CSV")


def convert(value: float, unit: str, target: str) -> float:
    # TODO(4): Convert time to microseconds (`us`), byte counts to decimal
    # megabytes (`mb`), and shared memory to decimal kilobytes (`kb`).
    #
    # Strip suffixes such as `/block` before conversion.
    # Time units: ns, us, ms, s
    # Data units: byte, Kbyte, Mbyte, Gbyte
    raise NotImplementedError("Day 06 TODO(4): normalize metric units")


def load_profile(path: Path, name: str) -> Profile:
    values, units = load_raw_csv(path)

    def metric(key: str, target: str | None = None) -> float:
        column = METRICS[key]
        value = parse_number(values[column])
        return convert(value, units[column], target) if target else value

    return Profile(
        name=name,
        kernel_name=values["Kernel Name"],
        block_size=values.get("Block Size", ""),
        grid_size=values.get("Grid Size", ""),
        duration_us=metric("duration", "us"),
        dram_read_mb=metric("dram_read", "mb"),
        dram_write_mb=metric("dram_write", "mb"),
        dram_throughput_pct=metric("dram_throughput"),
        global_load_requests=metric("global_load_requests"),
        global_store_requests=metric("global_store_requests"),
        global_load_sectors=metric("global_load_sectors"),
        global_store_sectors=metric("global_store_sectors"),
        load_sectors_per_request=metric("load_sectors_per_request"),
        store_sectors_per_request=metric("store_sectors_per_request"),
        load_bank_conflicts=metric("load_conflicts"),
        store_bank_conflicts=metric("store_conflicts"),
        occupancy_pct=metric("occupancy"),
        registers_per_thread=metric("registers"),
        dynamic_shared_kb=metric("shared_memory", "kb"),
    )


def compare(unpadded: Profile, padded: Profile) -> dict[str, float]:
    # TODO(5): First reject mismatched block/grid geometry or DRAM traffic
    # differing by more than 10%. Then return:
    #   speedup = unpadded duration / padded duration
    #   conflict_reduction_pct
    #   dram_throughput_delta_points
    #   occupancy_delta_points
    #   shared_memory_delta_bytes
    #
    # Percent conflict reduction is:
    #   100 * (unpadded_total - padded_total) / unpadded_total
    raise NotImplementedError("Day 06 TODO(5): compare profiles")


def compare_naive(read: Profile, write: Profile) -> dict[str, float]:
    # TODO(6): Verify matching launch geometry, then return:
    #   write_speedup = read duration / write duration
    #   read_load_sectors_per_request
    #   read_store_sectors_per_request
    #   write_load_sectors_per_request
    #   write_store_sectors_per_request
    #
    # These four sector ratios show that thread remapping moves the scattered
    # direction from stores to loads rather than eliminating it.
    raise NotImplementedError("Day 06 TODO(6): compare naive mappings")


def main() -> int:
    parser = argparse.ArgumentParser()
    fixtures = Path(__file__).with_name("fixtures")
    parser.add_argument(
        "--naive-read",
        type=Path,
        default=fixtures / "naive_read.csv",
    )
    parser.add_argument(
        "--naive-write",
        type=Path,
        default=fixtures / "naive_write.csv",
    )
    parser.add_argument(
        "--unpadded",
        type=Path,
        default=fixtures / "shared_unpadded.csv",
    )
    parser.add_argument(
        "--padded",
        type=Path,
        default=fixtures / "shared_padded.csv",
    )
    parser.add_argument(
        "--swizzled",
        type=Path,
        default=fixtures / "shared_swizzled.csv",
    )
    args = parser.parse_args()

    naive_read = load_profile(args.naive_read, "naive coalesced read")
    naive_write = load_profile(args.naive_write, "naive coalesced write")
    unpadded = load_profile(args.unpadded, "unpadded stride 32")
    padded = load_profile(args.padded, "padded stride 33")
    swizzled = load_profile(args.swizzled, "swizzled S<5,0,5>")

    # TODO(7): Print all five profiles and evidence-based diagnoses:
    #
    # Challenge A:
    #   - compare_naive shows the 4/32 versus 32/4 sector swap;
    #   - explain why contiguous stores beat scattered stores.
    #
    # Challenge B:
    #   - same intended workload,
    #   - much lower total bank conflicts,
    #   - lower duration / higher DRAM utilization,
    #   - nearly unchanged occupancy and registers.
    #
    # Compare unpadded with both padded and swizzled. Also report that
    # swizzling avoids padding's extra shared-memory allocation.
    print(
        compare_naive(naive_read, naive_write),
        compare(unpadded, padded),
        compare(unpadded, swizzled),
    )
    return 0

## 9. Run the completed reference analyzer

This cell uses the H100 fixture snapshots so it is deterministic
and does not require a live profiler.

In [ ]:
import importlib.util
import sys

spec = importlib.util.spec_from_file_location(
    "day06_solution", day_root / "solution.py"
)
solution = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = solution
spec.loader.exec_module(solution)

profiles = {
    name: solution.load_profile(path, name)
    for name, path in fixtures.items()
}
for profile in profiles.values():
    solution.print_profile(profile)
solution.print_naive_comparison(
    profiles["naive read"], profiles["naive write"]
)
solution.print_shared_comparison(
    profiles["shared unpadded"], profiles["shared padded"], "padding"
)
solution.print_shared_comparison(
    profiles["shared unpadded"],
    profiles["shared swizzled"],
    "swizzling",
)

### Analyze your five live reports

After running `collect_profiles.sh`, load Challenge A and Challenge
B together. `print_naive_comparison` reports the speedup and the
load/store sectors per request that identify the coalesced side.

In [ ]:
live_paths = {
    "naive read": reports_dir / "naive_read.csv",
    "naive write": reports_dir / "naive_write.csv",
    "shared unpadded": reports_dir / "shared_unpadded.csv",
    "shared padded": reports_dir / "shared_padded.csv",
    "shared swizzled": reports_dir / "shared_swizzled.csv",
}
if all(path.exists() for path in live_paths.values()):
    live = {
        name: solution.load_profile(path, name)
        for name, path in live_paths.items()
    }
    for profile in live.values():
        solution.print_profile(profile)
    solution.print_naive_comparison(
        live["naive read"], live["naive write"]
    )
else:
    print("Run collect_profiles.sh to create all five live reports.")

## Why coalesced write can be much faster

A warp's memory request is split into 32-byte sectors. Challenge A
moves the scattered direction rather than removing it:

```text
naive-read:  contiguous loads, scattered stores
naive-write: scattered loads, contiguous stores
```

NCU should show this swap in average sectors/request. Scattered
stores are often more expensive: their sectors must be merged and
committed through the cache/memory hierarchy, while strided loads
can benefit from cache-line fetch and reuse behavior. The exact
speedup is architecture- and shape-dependent, so treat sector
counts plus duration as the evidence rather than assuming symmetry.

## Why `S<5,0,5>` matches the Float32 tile

CuTe layouts map coordinates to **element offsets**. For a
row-major `32 x 32` tile:

```text
offset bits 0..4 = column
offset bits 5..9 = row
```

A Float32 word occupies one shared-memory bank unit, so the low
five element-offset bits select one of 32 banks. `S<5,0,5>` XORs
the five row bits into those five bank-selecting column bits. A
transposed warp read therefore spreads across the banks without
allocating a padded column.

## 10. Interpret the H100 evidence

The captured example shows approximately:

| Evidence | Unpadded | Padded | Swizzled |
| --- | ---: | ---: | ---: |
| Duration | 104.192 us | 44.256 us | 44.640 us |
| Total bank conflicts | 16.37 M | 35.3 K | 32.2 K |
| DRAM throughput | 33.60% | 77.73% | 77.21% |
| Occupancy | 93.20% | 92.12% | 92.12% |
| Registers/thread | 28 | 26 | 26 |
| Dynamic shared memory | 4.096 KB | 4.220 KB | 4.096 KB |

Challenge A's same capture shows:

| Variant | Duration | Load sectors/request | Store sectors/request |
| --- | ---: | ---: | ---: |
| Coalesced read | 283.872 us | 4 | 32 |
| Coalesced write | 88.864 us | 32 | 4 |

This supports the causal chain:

```text
stride 33
  -> lanes distribute across shared banks
  -> shared-load conflict counter collapses
  -> warps spend less time serializing shared reads
  -> DRAM pipeline stays busier
  -> kernel duration falls
```

Store conflicts need not move identically to load conflicts. The
transposed operation is the shared read, and total conflicts remain
dominated by the large load reduction.

## 11. Sanity checks before accepting a comparison

Reject or qualify the result if:

- block or grid dimensions differ;
- DRAM traffic differs substantially;
- more than one kernel launch appears;
- one report has unavailable metrics;
- clocks or concurrent workloads changed drastically;
- the profiled target failed correctness;
- different NCU metric sets caused different replay behavior.

Similar DRAM bytes do not need to be bit-identical because cache and
compression behavior can vary. They should nevertheless represent
the same order of work.

## 12. Profiling is not benchmarking

Use NCU to answer **why**. Use Day 05's `testing.benchmark` path to
answer **how fast in steady state**.

NCU instrumentation can alter clocks, replay kernels, flush caches,
and serialize execution. Its duration is valuable for controlled
within-report comparisons, not as a replacement for application
latency. A complete optimization result includes:

1. correctness on edge shapes;
2. steady-state benchmark with warmup;
3. targeted profiler evidence;
4. resource and regression checks.

## 13. Extension experiments

After padding, profile one change at a time:

1. **Different swizzles:** derive bit mappings, then compare conflicts.
2. **128-bit global copies:** inspect global sectors and duration.
3. **Different tile sizes:** compare occupancy, waves, and conflicts.
4. **Irregular shapes:** quantify predicate and partial-tile cost.
5. **Roofline/SOL sections:** determine whether the optimized kernel
   is now limited by DRAM, L2, L1/shared, or instruction issue.

Never begin with `--set full` unless you know why every section is
needed. Broad collection is slow and produces more data than a
focused hypothesis requires.

## What you should know afterward

- Form a metric prediction before opening the profiler.
- Compare controlled kernel variants with matching launch geometry.
- Filter NCU to the target kernel and request a small metric set.
- Treat raw CSV units as data, not decoration.
- Use mechanism counters, outcome metrics, and resource controls together.
- Distinguish NCU replay duration from steady-state benchmark time.
- Preserve `.ncu-rep` for interactive inspection and CSV for
  reproducible analysis.